# Climate Mitigation Potential widget data preparation

## Load libraries

In [3]:
import pandas as pd
import json


## Load and clean data
Mitigation potential data is stored in a CSV on the [GCS folder](https://console.cloud.google.com/storage/browser/wetlands-gap-map/original_data;tab=objects?inv=1&invt=Ab12FA&prefix=&forceOnObjectsSortingFiltering=false).  
It contains all countries, needs to be filtered to Sahel countries.  


In [ ]:
df = pd.read_csv('https://storage.googleapis.com/wetlands-gap-map/original_data/Mitigation_Country_clean.csv')
sel_cols = ['ISO',
        'Reduce deforestation AVG', 'Reforestation AVG', 'Forest management AVG',
       'Grassland and savanna fire mgmt ',
       'Reduce peatland degradation', 'Peatland restoration',
       'Reduce Mangrove conversion', 'Mangrove restoration'] # Selected columns need to be reviewed for final data
df = df[sel_cols]
df.columns = df.columns.str.replace(' ', '_').str.lower()
df.head()

,iso,reduce_deforestation_avg,reforestation_avg,forest_management_avg,grassland_and_savanna_fire_mgmt_,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration
0,AFG,54.7488,84.1559,100.6039,NaN,NaN,NaN,NaN,NaN
1,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ALB,NaN,54.0851,9.6573,NaN,NaN,NaN,NaN,NaN
3,DZA,NaN,126.3656,6.4040,NaN,NaN,NaN,NaN,NaN
4,ASM,NaN,547.1500,0.0000,NaN,NaN,NaN,NaN,NaN


In [ ]:
#Filter to Sahel countries
countries = pd.read_csv('https://storage.googleapis.com/wetlands-gap-map/original_data/countries_sahel.csv')
print(f'Initial countries: {len(df)}')
df = df[df['iso'].isin(countries['code'])]
print(f'Countries after filtering: {len(df)}')

Initial countries: 250
Countries after filtering: 22


## Process data.  
Process and reshape the data for match needed format.

In [30]:
# Pivot to long table and create location id column
df_long = df.melt(id_vars=['iso'], var_name='intervention', value_name='value')
df_long['location'] = df_long['iso'].apply(lambda x: f'adminregion-{x.lower()}')
df_long = df_long[df_long['value'].notna()]
df_long = df_long[['location', 'intervention', 'value']]
df_long

,location,intervention,value
0,adminregion-ben,reduce_deforestation_avg,262.9976
1,adminregion-bfa,reduce_deforestation_avg,115.6229
2,adminregion-cmr,reduce_deforestation_avg,374.3571
3,adminregion-caf,reduce_deforestation_avg,262.3109
4,adminregion-tcd,reduce_deforestation_avg,225.7684
...,...,...,...
165,adminregion-gnb,mangrove_restoration,704.0000
166,adminregion-ken,mangrove_restoration,704.0000
170,adminregion-nga,mangrove_restoration,704.0000
171,adminregion-sen,mangrove_restoration,704.0000


In [31]:
# Combine data for each country as dictionary, add indicator and id columns
df_location = df_long.groupby('location').apply(
    lambda x: dict(zip(x['intervention'], x['value']))
).reset_index(name='data')

df_location['id'] = 'mitigation-' + df_location.index.astype(str)
df_location['data'] = df_location['data'].apply(json.dumps)
df_location['indicator'] = 'wetlands-mitigation-potential'
df_location = df_location[['id', 'indicator', 'location', 'data']]
df_location.head()

/var/folders/wf/_wlxc6cn5js4hh3j7j6x28f80000gn/T/ipykernel_20910/2658583823.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_location = df_long.groupby('location').apply(


,id,indicator,location,data
0,mitigation-0,wetlands-mitigation-potential,adminregion-ben,"{""reduce_deforestation_avg"": 262.9976, ""refore..."
1,mitigation-1,wetlands-mitigation-potential,adminregion-bfa,"{""reduce_deforestation_avg"": 115.6229, ""refore..."
2,mitigation-2,wetlands-mitigation-potential,adminregion-caf,"{""reduce_deforestation_avg"": 262.3109, ""refore..."
3,mitigation-3,wetlands-mitigation-potential,adminregion-civ,"{""reduce_deforestation_avg"": 244.1175, ""refore..."
4,mitigation-4,wetlands-mitigation-potential,adminregion-cmr,"{""reduce_deforestation_avg"": 374.3571, ""refore..."


## Save data

In [32]:
df_location.to_csv('../data/processed/test_mitigation_indicator_data.csv', index=False)

In [33]:
df_location.to_json('../data/processed/test_mitigation_indicator_data.json', orient='records', lines=True)